# AURA — Online Learning & Model Management

This notebook demonstrates how to evolve the phishing detector on
live traffic using the `inference.OnlineLearner`, how to gate the
promotion of a new version, and how to manage versions through the
`ModelRegistry` (list, switch, roll back).

## The problem we are trying to fix

The production model (`v1_0`) is very strong at catching phishing,
but it is **aggressive against legitimate marketing / transactional
mail**:

- subscription renewal receipts
- payment confirmations
- flash-sale and promotional emails

These emails share surface features with phishing (urgency language,
money amounts, click-through URLs) and are frequently false-flagged.
We will:

1. Generate a **test set** of aggressive marketing / payment /
   subscription emails (plus a few phishing controls) and measure the
   baseline `v1_0` accuracy on it.
2. Generate a **training batch** from the same distribution with
   correct labels, plus a **holdout** for honest before/after metrics.
3. Run `OnlineLearner.partial_fit_batch` to register a new version.
4. Promote the new version through the gated `promote(...)` API.
5. Re-score the **same test set** with the new model version and
   measure the improvement.
6. Demonstrate model management: list versions, switch active version,
   and roll back.

Only the public API from `inference/USAGE.md` is used.
For the prediction-only walkthrough, see `demo_predict.ipynb`.

## 1. Setup

Import the public API, configure `aura.inference` logging at INFO,
and load the `v1_0` detector explicitly so we can compare it against
the post-fine-tune version later.

In [1]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('aura.inference').setLevel(logging.INFO)

repo_root = Path.cwd()
for _ in range(4):
    if (repo_root / 'inference' / '__init__.py').exists():
        break
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.environ.setdefault('AURA_MODELS_DIR', str((repo_root / 'models').resolve()))

from inference import (
    ModelRegistry,
    OnlineLearner,
    PhishingDetector,
    ValidationError,
)

registry = ModelRegistry(os.environ['AURA_MODELS_DIR'])
BASELINE_VERSION = 'v1_0'

detector_v1 = PhishingDetector.load(BASELINE_VERSION)
print('baseline version :', detector_v1.version)
print('registry versions:', registry.list_versions())
print('active version   :', registry.active_version())

INFO aura.inference: loading model version=v1_0 from C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\v1_0\production\phishing_detector_mlp_classifier.pkl


baseline version : v1_0
registry versions: ['v1_0']
active version   : None


## 2. Synthetic data

We build three disjoint sets from a small template library covering:

- **payment** confirmations / receipts
- **subscription** renewals
- **promotional / flash-sale** marketing
- a handful of clear **phishing** controls so we can verify that
  fine-tuning does not regress on the phishing class

All marketing emails use real, well-known brand domains so that any
feature the model keys on is genuinely about the email content — not
sender-domain forgery.

In [2]:
import random
rng = random.Random(17)

BRANDS = [
    ('Amazon',   'amazon.com'),
    ('Netflix',  'netflix.com'),
    ('Spotify',  'spotify.com'),
    ('Apple',    'apple.com'),
    ('Stripe',   'stripe.com'),
    ('Adobe',    'adobe.com'),
    ('Dropbox',  'dropbox.com'),
    ('Best Buy', 'bestbuy.com'),
    ('Shopify',  'shopify.com'),
    ('Uber',     'uber.com'),
]

# (category, sender_t, subject_t, body_t)
MARKETING_TEMPLATES = [
    ('payment',
     '"{brand}" <receipts@{domain}>',
     'Your receipt from {brand} - Order #{order}',
     'Hi there, thanks for your payment of ${amount}. Your receipt is attached. '
     'Manage your subscription or cancel anytime at https://billing.{domain}/p/login. '
     'If you did not make this purchase, please contact support.'),
    ('payment',
     '"{brand}" <no-reply@{domain}>',
     'Payment confirmation: ${amount} charged to Visa ****{card}',
     'We have successfully charged ${amount} to the card ending in {card}. '
     'View the invoice at https://www.{domain}/invoices/{order}. Thanks for your business.'),
    ('subscription',
     '"{brand}" <subscriptions@{domain}>',
     'Your {brand} subscription renews on {date}',
     'Your {brand} subscription has been renewed for another month. '
     'Your next billing date is {date}. Update payment details at '
     'https://www.{domain}/account.'),
    ('subscription',
     '"{brand}" <billing@{domain}>',
     'Reminder: {brand} Premium renews in 3 days',
     'Just a heads-up - your {brand} Premium subscription will renew on {date} '
     'for ${amount}. No action needed. Manage your plan at '
     'https://www.{domain}/account/subscription.'),
    ('promo',
     '"{brand} Deals" <deals@{domain}>',
     'FLASH SALE: {discount}% off everything - ends tonight!!!',
     'Limited time offer! Save {discount}% on all items today only. '
     'Shop now at https://www.{domain}/sale before it ends tonight. '
     'Click here to see your personalised deals. Hurry!'),
    ('promo',
     '"{brand}" <offers@{domain}>',
     'Last chance! Your {discount}% coupon expires soon',
     'Do not miss out - your exclusive {discount}% off coupon expires on {date}. '
     'Redeem now at https://www.{domain}/coupon?code=SAVE{discount}. '
     'One-time use only. Terms apply.'),
    ('shipping',
     '"{brand}" <orders@{domain}>',
     'Your {brand} order has shipped',
     'Your order #{order} has shipped and is on its way. '
     'Track it at https://www.{domain}/orders/{order}. '
     'Estimated delivery: {date}.'),
]

PHISH_TEMPLATES = [
    ('"PayPal Security" <service@paypa1-alerts.com>',
     'URGENT: Your account will be suspended in 24 hours!!!',
     'Dear customer, we detected suspicious activity on your account. '
     'Please verify your credentials at http://paypa1-alerts.com/verify '
     'within 24 hours or your account will be locked!!!'),
    ('"Microsoft 365" <no-reply@ms-support-verify.net>',
     'Action required: verify your identity immediately',
     'Your mailbox is over quota. Click http://ms-support-verify.net/login '
     'to re-authenticate NOW, otherwise incoming mail will bounce.'),
    ('"DHL Express" <tracking@dhl-delivery-notice.info>',
     'Package delivery failed - reschedule within 12 hours',
     'Your DHL package could not be delivered. Confirm address at '
     'http://dhl-delivery-notice.info/track?id=982137 before it is returned.'),
    ('"Chase Bank" <alerts@secure-chase-online.com>',
     'Unusual sign-in attempt detected - confirm now',
     'Final notice: update payment details at '
     'http://secure-chase-online.com/account/update to avoid service disruption.'),
    ('"HR Department" <hr.payroll@company-benefits.co>',
     'Payroll update: click to review your new paystub',
     'HR has posted an updated paystub. Download here: '
     'http://company-benefits.co/hr/paystub.pdf - password is your employee ID.'),
]

DATES = ['April 20', 'April 30', 'May 5', 'May 15', 'June 1', 'June 10']

def make_marketing(rng):
    brand, domain = rng.choice(BRANDS)
    cat, sender_t, subject_t, body_t = rng.choice(MARKETING_TEMPLATES)
    kw = {
        'brand':    brand,
        'domain':   domain,
        'order':    f'{rng.randint(10000, 99999)}',
        'amount':   f'{rng.uniform(4.99, 199.99):.2f}',
        'card':     f'{rng.randint(1000, 9999)}',
        'date':     rng.choice(DATES),
        'discount': rng.choice([20, 30, 40, 50, 60, 70]),
    }
    return {
        'category': cat,
        'sender':   sender_t.format(**kw),
        'subject':  subject_t.format(**kw),
        'body':     body_t.format(**kw),
        'label':    0,
    }

def make_phishing(rng):
    sender, subject, body = rng.choice(PHISH_TEMPLATES)
    return {
        'category': 'phishing',
        'sender':   sender,
        'subject':  subject,
        'body':     body,
        'label':    1,
    }

def make_mixed(rng, n_marketing: int, n_phishing: int):
    rows = [make_marketing(rng) for _ in range(n_marketing)] \
         + [make_phishing(rng)  for _ in range(n_phishing)]
    rng.shuffle(rows)
    return rows

# Three disjoint draws from the same distribution.
test_set     = make_mixed(rng, n_marketing=25, n_phishing=5)
train_batch  = make_mixed(rng, n_marketing=40, n_phishing=10)
holdout_rows = make_mixed(rng, n_marketing=15, n_phishing=5)

print(f'test_set    : {len(test_set)} rows '
      f'({sum(1 for r in test_set if r["label"] == 0)} legit, '
      f'{sum(1 for r in test_set if r["label"] == 1)} phishing)')
print(f'train_batch : {len(train_batch)} rows '
      f'({sum(1 for r in train_batch if r["label"] == 0)} legit, '
      f'{sum(1 for r in train_batch if r["label"] == 1)} phishing)')
print(f'holdout     : {len(holdout_rows)} rows '
      f'({sum(1 for r in holdout_rows if r["label"] == 0)} legit, '
      f'{sum(1 for r in holdout_rows if r["label"] == 1)} phishing)')

test_set    : 30 rows (25 legit, 5 phishing)
train_batch : 50 rows (40 legit, 10 phishing)
holdout     : 20 rows (15 legit, 5 phishing)


### Peek at a few generated emails

In [3]:
peek = pd.DataFrame([
    {'category': r['category'], 'label': r['label'], 'sender': r['sender'],
     'subject': r['subject'][:70]}
    for r in test_set[:8]
])
peek

,category,label,sender,subject
0,promo,0,"""Netflix Deals"" <deals@netflix.com>",FLASH SALE: 20% off everything - ends tonight!!!
1,phishing,1,"""HR Department"" <hr.payroll@company-benefits.co>",Payroll update: click to review your new paystub
2,promo,0,"""Netflix Deals"" <deals@netflix.com>",FLASH SALE: 20% off everything - ends tonight!!!
3,payment,0,"""Stripe"" <receipts@stripe.com>",Your receipt from Stripe - Order #46108
4,promo,0,"""Shopify"" <offers@shopify.com>",Last chance! Your 50% coupon expires soon
5,promo,0,"""Stripe"" <offers@stripe.com>",Last chance! Your 40% coupon expires soon
6,phishing,1,"""Chase Bank"" <alerts@secure-chase-online.com>",Unusual sign-in attempt detected - confirm now
7,promo,0,"""Shopify"" <offers@shopify.com>",Last chance! Your 30% coupon expires soon


## 3. Baseline evaluation on `v1_0`

Run the **same test set** through the baseline detector and break the
results down by category. Anything flagged as phishing when
`label == 0` is a false positive — this is the behaviour we want to
improve.

In [4]:
def score(detector, rows):
    emails = [{k: r[k] for k in ('sender', 'subject', 'body')} for r in rows]
    results = detector.predict_batch(emails, threshold=0.5)
    return pd.DataFrame({
        'category':    [r['category'] for r in rows],
        'true_label':  [r['label'] for r in rows],
        'pred_label':  [p.predicted_label for p in results],
        'phish_prob':  [round(p.phishing_probability, 4) for p in results],
        'subject':     [r['subject'][:60] for r in rows],
    })

baseline_df = score(detector_v1, test_set)

def summarise(df, label):
    overall = (df['true_label'] == df['pred_label']).mean()
    legit = df[df['true_label'] == 0]
    phish = df[df['true_label'] == 1]
    fp_rate = (legit['pred_label'] == 1).mean() if len(legit) else float('nan')
    recall  = (phish['pred_label'] == 1).mean() if len(phish) else float('nan')
    print(f'[{label}] accuracy={overall:.2%}  '
          f'false_positive_rate={fp_rate:.2%}  phishing_recall={recall:.2%}')

summarise(baseline_df, 'baseline v1_0')

print('\nFalse positives (legit emails flagged as phishing):')
baseline_df[(baseline_df['true_label'] == 0) & (baseline_df['pred_label'] == 1)]

[baseline v1_0] accuracy=63.33%  false_positive_rate=44.00%  phishing_recall=100.00%

False positives (legit emails flagged as phishing):


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


,category,true_label,pred_label,phish_prob,subject
0,promo,0,1,0.9314,FLASH SALE: 20% off everything - ends tonight!!!
2,promo,0,1,0.9314,FLASH SALE: 20% off everything - ends tonight!!!
4,promo,0,1,0.9283,Last chance! Your 50% coupon expires soon
5,promo,0,1,0.7140,Last chance! Your 40% coupon expires soon
7,promo,0,1,0.7944,Last chance! Your 30% coupon expires soon
12,promo,0,1,0.9260,FLASH SALE: 30% off everything - ends tonight!!!
18,promo,0,1,0.9802,FLASH SALE: 70% off everything - ends tonight!!!
19,promo,0,1,0.9656,FLASH SALE: 50% off everything - ends tonight!!!
20,promo,0,1,0.8778,FLASH SALE: 60% off everything - ends tonight!!!
21,promo,0,1,0.9697,FLASH SALE: 50% off everything - ends tonight!!!


Breakdown by category — where the false positives cluster:

In [5]:
baseline_df.assign(correct=baseline_df['true_label'] == baseline_df['pred_label']) \
           .groupby('category') \
           .agg(n=('category', 'size'),
                accuracy=('correct', 'mean'),
                mean_phish_prob=('phish_prob', 'mean')) \
           .round(4)

,n,accuracy,mean_phish_prob
category,,,
payment,6,1.0000,0.0072
phishing,5,1.0000,0.9984
promo,10,0.0000,0.9019
subscription,9,0.8889,0.1511


## 4. Online learning

Feed the correctly-labelled training batch into `OnlineLearner`. The
learner:

- clones the source model (no in-place mutation)
- runs capped `partial_fit` iterations on the batch
- scores before/after on the supplied **holdout** for an honest signal
- **registers** a new version in the registry, but does not promote
  it

The holdout set is a separate disjoint draw from the same
distribution, so the reported metrics are not contaminated by the
training batch. Wrapped in `try/except` so the notebook degrades
gracefully if artefacts are missing.

In [6]:
holdout_df = pd.DataFrame([
    {'sender': r['sender'], 'subject': r['subject'], 'body': r['body']}
    for r in holdout_rows
])
holdout_y = np.array([r['label'] for r in holdout_rows], dtype=np.int64)

learner = OnlineLearner(registry, holdout_set=(holdout_df, holdout_y))

online_result = None
try:
    online_result = learner.partial_fit_batch(
        [{k: r[k] for k in ('sender', 'subject', 'body', 'label')} for r in train_batch],
        source_version=BASELINE_VERSION,
        max_iter_per_call=15,
    )
    print('new_version         :', online_result.new_version)
    print('source_version      :', online_result.source_version)
    print('batch_size          :', online_result.batch_size)
    print('iterations          :', online_result.iterations)
    print('performance_before  :', online_result.performance_before)
    print('performance_after   :', online_result.performance_after)
    print('oov_rate_subject    :', round(online_result.oov_rate_subject, 4))
    print('oov_rate_body       :', round(online_result.oov_rate_body, 4))
except (ValidationError, FileNotFoundError) as e:
    print('online learning skipped:', e)

WARNING aura.inference: subject OOV rate 0.325 exceeds threshold 0.300


WARNING aura.inference: body OOV rate 0.535 exceeds threshold 0.300


Iteration 1, loss = 2.06413337
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
Iteration 1, loss = 0.41406800
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


Iteration 1, loss = 0.36396162
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
Iteration 1, loss = 0.03414339
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


Iteration 1, loss = 0.03442135
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
Iteration 1, loss = 0.04034622
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


...

Iteration 1, loss = 0.03381928
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
Iteration 1, loss = 0.03359501
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


Iteration 1, loss = 0.03350408
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
Iteration 1, loss = 0.03344628
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


INFO aura.inference: registered new_version=v1_1 (source=v1_0)


Iteration 1, loss = 0.03339816
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


new_version         : v1_1
source_version      : v1_0
batch_size          : 50
iterations          : 15
performance_before  : {'accuracy': 0.8, 'precision': 0.5555555555555556, 'recall': 1.0, 'f1': 0.7142857142857143}
performance_after   : {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
oov_rate_subject    : 0.3254
oov_rate_body       : 0.5351


## 5. Gated promotion

Promotion is a separate, gated step. `promote(..., min_delta_f1=...)`
refuses to flip the active-version pointer if the new version's
holdout F1 regressed by more than the supplied tolerance.

We set a relatively lenient tolerance here because the existing model
already saturates F1 on obvious phishing — the wins we expect from
this round are mostly in **specificity** on marketing/payment mail,
which will show up as a slightly different F1 rather than a large
jump.

In [7]:
promoted_version = None
if online_result is None:
    print('skipping promotion - no online-learning result available')
else:
    try:
        learner.promote(online_result.new_version, min_delta_f1=-0.05)
        promoted_version = online_result.new_version
        print('promoted          :', promoted_version)
    except ValidationError as e:
        print('refused to promote:', e)

    print('active_version now:', registry.active_version())
    print('all versions      :', registry.list_versions())

INFO aura.inference: promoted version=v1_1 metrics={'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


promoted          : v1_1
active_version now: v1_1
all versions      : ['v1_0', 'v1_1']


## 6. Re-score the same test set on the new version

Load the newly-promoted version explicitly, run the **same** test set
through it, and compare against the baseline numbers.

In [8]:
if promoted_version is None:
    print('no promoted version available; skipping after-scoring')
else:
    detector_v2 = PhishingDetector.load(promoted_version)
    after_df = score(detector_v2, test_set)

    print('Before vs. after on the identical test set:')
    summarise(baseline_df, f'before ({BASELINE_VERSION})')
    summarise(after_df,    f'after  ({promoted_version})')

    compare = baseline_df[['category', 'true_label', 'subject']].copy()
    compare['pred_before'] = baseline_df['pred_label']
    compare['prob_before'] = baseline_df['phish_prob']
    compare['pred_after']  = after_df['pred_label']
    compare['prob_after']  = after_df['phish_prob']
    compare['flipped']     = compare['pred_before'] != compare['pred_after']
    display(compare)

INFO aura.inference: loading model version=v1_1 from C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\v1_1\production\phishing_detector_mlp_classifier.pkl


Before vs. after on the identical test set:
[before (v1_0)] accuracy=63.33%  false_positive_rate=44.00%  phishing_recall=100.00%
[after  (v1_1)] accuracy=100.00%  false_positive_rate=0.00%  phishing_recall=100.00%


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


,category,true_label,subject,pred_before,prob_before,pred_after,prob_after,flipped
0,promo,0,FLASH SALE: 20% off everything - ends tonight!!!,1,0.9314,0,0.0028,True
1,phishing,1,Payroll update: click to review your new paystub,1,0.9989,1,0.9948,False
2,promo,0,FLASH SALE: 20% off everything - ends tonight!!!,1,0.9314,0,0.0028,True
3,payment,0,Your receipt from Stripe - Order #46108,0,0.0106,0,0.0023,False
4,promo,0,Last chance! Your 50% coupon expires soon,1,0.9283,0,0.0052,True
5,promo,0,Last chance! Your 40% coupon expires soon,1,0.7140,0,0.0024,True
6,phishing,1,Unusual sign-in attempt detected - confirm now,1,0.9992,1,0.9970,False
7,promo,0,Last chance! Your 30% coupon expires soon,1,0.7944,0,0.0031,True
8,payment,0,Payment confirmation: $66.12 charged to Visa *...,0,0.0057,0,0.0009,False
9,subscription,0,Your Apple subscription renews on April 20,0,0.0078,0,0.0005,False


### Per-category improvement

In [9]:
if promoted_version is not None:
    before_acc = baseline_df.assign(ok=baseline_df['true_label'] == baseline_df['pred_label']) \
                            .groupby('category')['ok'].mean()
    after_acc  = after_df.assign(ok=after_df['true_label'] == after_df['pred_label']) \
                         .groupby('category')['ok'].mean()
    delta = pd.DataFrame({
        'before_accuracy': before_acc.round(4),
        'after_accuracy':  after_acc.round(4),
    })
    delta['delta'] = (delta['after_accuracy'] - delta['before_accuracy']).round(4)
    display(delta)

,before_accuracy,after_accuracy,delta
category,,,
payment,1.0000,1.0,0.0000
phishing,1.0000,1.0,0.0000
promo,0.0000,1.0,1.0000
subscription,0.8889,1.0,0.1111


## 7. Model management — switching between versions

The registry is the source of truth for which version is active.
`set_active(version)` performs an **atomic** metadata write and
verifies the model's `sha256` against the registry record before
switching. `load_production()` resolves to whatever is active at the
moment of the call.

In [10]:
print('all versions     :', registry.list_versions())
print('active_version() :', registry.active_version())
print('latest_version() :', registry.latest_version())

for v in registry.list_versions():
    print(f'\npaths_for({v!r}):')
    for k, p in registry.paths_for(v).items():
        print(f'  {k:22s} {p}')

all versions     : ['v1_0', 'v1_1']
active_version() : v1_1
latest_version() : v1_1

paths_for('v1_0'):
  model                  C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\v1_0\production\phishing_detector_mlp_classifier.pkl
  subject_vectorizer     C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\pipeline_components\subject_vectorizer.pkl
  body_vectorizer        C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\pipeline_components\body_vectorizer.pkl
  calibrator             C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\pipeline_components\calibrator.pkl

paths_for('v1_1'):
  model                  C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\v1_1\production\phishing_detector_mlp_classifier.pkl
  subject_vectorizer     C:\Users\Administrator\Docu

### Roll back to `v1_0`, then forward again

If something goes wrong in production, rolling back is one call.
`PhishingDetector.load_production()` after the rollback resolves to
`v1_0` again.

In [11]:
try:
    registry.set_active(BASELINE_VERSION)
    print('rolled back      :', registry.active_version())
    reloaded = PhishingDetector.load_production()
    print('load_production ->', reloaded.version)

    if promoted_version is not None:
        registry.set_active(promoted_version)
        print('rolled forward   :', registry.active_version())
        reloaded = PhishingDetector.load_production()
        print('load_production ->', reloaded.version)
except (ValueError, ValidationError) as e:
    print('model management error:', e)

INFO aura.inference: active_version set to v1_0


INFO aura.inference: loading model version=v1_0 from C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\v1_0\production\phishing_detector_mlp_classifier.pkl


rolled back      : v1_0


load_production -> v1_0


INFO aura.inference: active_version set to v1_1


INFO aura.inference: loading model version=v1_1 from C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\v1_1\production\phishing_detector_mlp_classifier.pkl


rolled forward   : v1_1
load_production -> v1_1


## 8. Cleanup notes

Running section 4 created a new version directory under `models/`
(for example `models/v1_1/production/`) containing the updated model
artefact and its `model_metadata.json`. Section 5 may also have
updated `models/model_metadata.json` to point `active_version` at the
new version.

To **roll back permanently** to the previous version:

```python
from inference import ModelRegistry
registry = ModelRegistry('models')
registry.set_active('v1_0')   # atomic; verifies sha256 by default
```

Deleting a new version directory from disk is safe *only* after
pointing `active_version` back at a known-good version — otherwise
the next call to `PhishingDetector.load_production()` will fail when
it tries to resolve the missing artefacts.

**Re-running this notebook** creates another version each time
(`v1_1`, then `v1_2`, …). That is expected: every online-learning
round is tracked as a distinct, promotable artefact. Clean them up
periodically if you do not want the registry to grow unbounded.